# Nemotron 0.86 Tinker Adapter Guide

This notebook is a practical public baseline for the **NVIDIA Nemotron Model Reasoning Challenge**. It does one thing carefully: it turns a public LoRA adapter into the exact `submission.zip` package expected by the competition evaluator, then verifies the package before you submit it.

The public score reached by this package was **0.86**. That was not achieved by a complicated local trick. The durable lesson is simpler: in this competition, adapter schema, namespace, zip layout, and provenance matter more than a long notebook that silently saves the wrong files.

## What this notebook is for

Use this notebook if you want a clean, reproducible reference for:

- finding a public adapter attached as a Kaggle model input,
- checking that it has the two required files,
- inspecting the LoRA config and safetensors header without loading a 30B model,
- creating a root-level `submission.zip`, and
- avoiding common failed-submission patterns.

It intentionally does **not** need internet or GPU. The adapter is treated as a submission artifact, not as a model to load for inference.

## Provenance and credit

The strongest public package used here comes from the public Kaggle model source:

`kienngx/nemotron-nano-30b-trained/Triton/tinker-adapter/1`

Credit goes to the original publisher of that adapter. This notebook adds a careful packaging, audit, and decision guide around the public artifact so that others can reproduce the submission shape and understand why several tempting alternatives underperformed.

## Public score evidence

These were the most useful score anchors from my public-score audit trail:

| Candidate family | Public score | Lesson |
|---|---:|---|
| Kien Tinker adapter package | 0.86 | Current public baseline to protect |
| Huikang v20 converted adapter | 0.85 | Valid package, but below the best |
| Sluitel broad adapter with best `lm_head` fill | 0.62 | Structural compatibility alone is not enough |
| Error1249x broad adapter with best `lm_head` fill | 0.59 | Another valid but task-misaligned broad adapter |
| Rick v41 full adapter replacement | 0.57 | Full public replacements can collapse |
| BharatMohan v31 full adapter replacement | 0.62 | Same warning as above |
| Tiny positive public-donor grafts | 0.86 | Some perturbations preserve score, but did not improve it |
| Takeshi verified-focus positive grafts | 0.86 | Sparse inproj/attention direction tied but did not beat 0.86 |

The point is not that `0.86` is unbeatable. The point is that the best-known public baseline is surprisingly easy to damage, so each candidate should be audited before spending a daily submission slot.

## Step 1: locate the adapter files

The competition submission is an adapter package. A naked CSV is not the real artifact for this challenge. The zip must contain, at minimum, these two files at the **zip root**:

- `adapter_config.json`
- `adapter_model.safetensors`

The cell below searches `/kaggle/input` for directories containing both files and prefers paths that look like a Tinker adapter.

In [ ]:
from pathlib import Path
import json
import os
import zipfile
from collections import Counter

INPUT_ROOT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
REQUIRED = ['adapter_config.json', 'adapter_model.safetensors']


def find_adapter_dirs(root=INPUT_ROOT):
    candidates = []
    for config_path in root.rglob('adapter_config.json'):
        adapter_dir = config_path.parent
        if all((adapter_dir / name).exists() for name in REQUIRED):
            model_size = (adapter_dir / 'adapter_model.safetensors').stat().st_size
            candidates.append((adapter_dir, model_size))
    return sorted(candidates, key=lambda item: ('tinker' not in str(item[0]).lower(), -item[1]))

candidates = find_adapter_dirs()
if not candidates:
    raise FileNotFoundError('No adapter directory with adapter_config.json and adapter_model.safetensors was found under /kaggle/input')

for i, (path, size) in enumerate(candidates, start=1):
    print(f'{i:02d}. {path}  size={size / (1024**3):.3f} GiB')

ADAPTER_DIR = candidates[0][0]
print('\nSelected adapter directory:')
print(ADAPTER_DIR)

## Step 2: inspect the adapter config

For this baseline, the important signs are a broad LoRA target set, rank 32, alpha 32, and root-level adapter files. The target modules include both attention projections and expert/feed-forward projections, plus `lm_head`.

In [ ]:
config_path = ADAPTER_DIR / 'adapter_config.json'
config = json.loads(config_path.read_text())

summary = {
    'peft_type': config.get('peft_type'),
    'r': config.get('r'),
    'lora_alpha': config.get('lora_alpha'),
    'lora_dropout': config.get('lora_dropout'),
    'target_modules': config.get('target_modules'),
    'task_type': config.get('task_type'),
}

print(json.dumps(summary, indent=2, sort_keys=True))

## Step 3: inspect the safetensors header without loading the model

The adapter file is large, but we can cheaply read only the safetensors header. This catches many bad candidates before upload: missing LoRA tensors, narrow target coverage, extra nesting, or a path that points to the wrong artifact.

In [ ]:
def read_safetensors_header(path):
    with path.open('rb') as f:
        header_len = int.from_bytes(f.read(8), 'little')
        header = json.loads(f.read(header_len))
    return header

model_path = ADAPTER_DIR / 'adapter_model.safetensors'
header = read_safetensors_header(model_path)
tensor_names = [name for name in header if name != '__metadata__']

print('Tensor count:', len(tensor_names))
print('Adapter size GiB:', round(model_path.stat().st_size / (1024**3), 3))
print('First 5 tensor names:')
for name in tensor_names[:5]:
    print('  ', name)

modules = config.get('target_modules') or []
module_counts = {}
for module in modules:
    marker = f'.{module}.'
    module_counts[module] = sum(marker in name for name in tensor_names)

print('\nApproximate tensor counts by target module marker:')
for module, count in sorted(module_counts.items()):
    print(f'{module:>10s}: {count}')

## Step 4: build `submission.zip`

The archive must place `adapter_config.json` and `adapter_model.safetensors` at the zip root. Do not put them inside an extra folder. `allowZip64=True` is required for multi-GB adapter packages.

In [ ]:
OUTPUT_ZIP = WORKING / 'submission.zip'
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

files_to_zip = [ADAPTER_DIR / name for name in REQUIRED]

with zipfile.ZipFile(
    OUTPUT_ZIP,
    'w',
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True,
) as zf:
    for path in files_to_zip:
        zf.write(path, arcname=path.name)

print('Created:', OUTPUT_ZIP)
print('Size GiB:', round(OUTPUT_ZIP.stat().st_size / (1024**3), 3))

## Step 5: verify the zip layout

This is the last cheap check before submission. The two required files should be exactly at the root.

In [ ]:
with zipfile.ZipFile(OUTPUT_ZIP) as zf:
    names = zf.namelist()
    infos = {info.filename: info.file_size for info in zf.infolist()}

print(names)
missing = [name for name in REQUIRED if name not in names]
if missing:
    raise RuntimeError(f'Missing required files from zip root: {missing}')

nested = [name for name in names if '/' in name.strip('/')]
if nested:
    raise RuntimeError(f'Unexpected nested file paths in submission zip: {nested[:5]}')

for name in REQUIRED:
    print(f'{name}: {infos[name] / (1024**3):.3f} GiB')

print('\nReady for Kaggle submission.')

## Why I recommend this as the baseline

A strong competition workflow starts by protecting the current best. In this challenge, many candidates look plausible because they are valid LoRA packages, but valid packages can still be very poor public submissions.

What helped most:

- Treat `submission.zip` shape as a hard contract.
- Inspect adapter namespace and target coverage before upload.
- Compare candidate tensors against the best known adapter before spending quota.
- Use small, information-bearing public-score probes instead of blind full replacement.
- Stop a branch when repeated public scores show it only ties or hurts.

What did not help:

- Submitting CSV-style outputs.
- Trusting notebook titles without checking actual output files.
- Replacing the whole adapter with a structurally compatible but task-misaligned public adapter.
- Over-interpreting tiny tensor perturbations when the public score is coarse.

## A reusable audit pattern for new adapters

For future candidates, I use this decision checklist before spending a daily submission slot:

1. Does the candidate produce a real `adapter_config.json` and `adapter_model.safetensors`?
2. Are the files at the correct root layout after zipping?
3. Does the config target the same broad module family as the best package?
4. Are the tensor names in the expected namespace?
5. If tensors are missing, is the fill strategy explicit and defensible?
6. Is the candidate genuinely new, or just a duplicate of a known 0.85/0.86 family?
7. Does the public-score history justify using one of the limited daily submissions?

That process is less exciting than a giant training notebook, but it prevents most wasted submissions.

## Closing note

If this notebook saves you a failed upload or gives you a clean starting point, please consider upvoting it and also checking the original public adapter source. The best next step is not to keep resubmitting this same package; it is to use this package as a stable baseline while searching for a genuinely new data or adapter signal that can move beyond 0.86.